# 03. ML을 위한 미적분 복습

## 학습 목표
- 도함수와 편미분의 기하학적 의미 이해
- Gradient와 Gradient Descent를 시각적으로 이해
- Chain Rule이 역전파(Backpropagation)의 수학적 기초임을 확인
- 수치 미분으로 해석적 미분의 정확성을 검증하는 방법 습득

## 참고 자료
- [3Blue1Brown - Essence of Calculus](https://www.youtube.com/playlist?list=PLZHQObOWTQDMsr9K-rj53DwVRMYO3t5Yr)
- [3Blue1Brown - Backpropagation](https://www.youtube.com/watch?v=Ilg3gGewQ5U)

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D

## 1. 미분 기초

### 1.1 도함수의 기하학적 의미

도함수 = 함수의 **순간 변화율** = 접선의 기울기

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

**ML에서의 의미**: 파라미터를 아주 조금 바꿨을 때, 손실(loss)이 얼마나 변하는가?

In [ ]:
# f(x) = x^2의 도함수 시각화
def f(x):
    return x ** 2

def f_prime(x):
    return 2 * x

x = np.linspace(-3, 3, 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 함수와 접선
ax = axes[0]
ax.plot(x, f(x), 'b-', linewidth=2, label='f(x) = x^2')

# 여러 점에서의 접선
for x0, color in [(-2, 'red'), (0, 'green'), (1.5, 'orange')]:
    slope = f_prime(x0)
    tangent = f(x0) + slope * (x - x0)
    ax.plot(x, tangent, '--', color=color, linewidth=1.5,
            label=f"tangent at x={x0} (slope={slope})")
    ax.plot(x0, f(x0), 'o', color=color, markersize=8)

ax.set_xlim(-3, 3)
ax.set_ylim(-2, 10)
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.set_title('Function and Tangent Lines')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 오른쪽: 도함수
ax = axes[1]
ax.plot(x, f(x), 'b-', linewidth=2, label="f(x) = x^2")
ax.plot(x, f_prime(x), 'r-', linewidth=2, label="f'(x) = 2x")
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Function vs Derivative')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("도함수 f'(x) = 2x:")
print("  x < 0: 기울기 음수 (함수 감소 중)")
print("  x = 0: 기울기 0 (극솟값!)")
print("  x > 0: 기울기 양수 (함수 증가 중)")

### 1.2 주요 미분 공식

ML에서 자주 쓰이는 함수들의 도함수:

| 함수 $f(x)$ | 도함수 $f'(x)$ | ML에서의 용도 |
|---|---|---|
| $x^n$ | $nx^{n-1}$ | 다항식 모델 |
| $e^x$ | $e^x$ | Softmax, 가중치 감쇠 |
| $\ln(x)$ | $1/x$ | Cross-entropy loss |
| $\sigma(x) = 1/(1+e^{-x})$ | $\sigma(x)(1-\sigma(x))$ | Sigmoid 활성화 |
| $\tanh(x)$ | $1 - \tanh^2(x)$ | Tanh 활성화 |

In [ ]:
# 활성화 함수와 그 도함수
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_deriv(x):
    s = sigmoid(x)
    return s * (1 - s)

def relu(x):
    return np.maximum(0, x)

def relu_deriv(x):
    return (x > 0).astype(float)

x = np.linspace(-5, 5, 200)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Sigmoid
ax = axes[0]
ax.plot(x, sigmoid(x), 'b-', linewidth=2, label='sigmoid(x)')
ax.plot(x, sigmoid_deriv(x), 'r--', linewidth=2, label="sigmoid'(x)")
ax.set_title('Sigmoid and its Derivative')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(0, color='black', linewidth=0.5)

# Tanh
ax = axes[1]
ax.plot(x, np.tanh(x), 'b-', linewidth=2, label='tanh(x)')
ax.plot(x, 1 - np.tanh(x)**2, 'r--', linewidth=2, label="tanh'(x)")
ax.set_title('Tanh and its Derivative')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(0, color='black', linewidth=0.5)

# ReLU
ax = axes[2]
ax.plot(x, relu(x), 'b-', linewidth=2, label='ReLU(x)')
ax.plot(x, relu_deriv(x), 'r--', linewidth=2, label="ReLU'(x)")
ax.set_title('ReLU and its Derivative')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

print("Sigmoid: 도함수 최대 0.25 -> Vanishing Gradient 문제의 원인")
print("ReLU: 양수 구간에서 도함수 = 1 -> Gradient가 잘 전달됨 (현대 딥러닝의 기본)")

### 1.3 편미분 (Partial Derivative)

다변수 함수에서 **하나의 변수만** 변화시키고 나머지는 고정.

$$f(x, y) = x^2 + 3xy + y^2$$

$$\frac{\partial f}{\partial x} = 2x + 3y \quad \text{(y는 상수 취급)}$$

$$\frac{\partial f}{\partial y} = 3x + 2y \quad \text{(x는 상수 취급)}$$

**ML에서**: 손실 함수는 수백만 개의 파라미터에 대한 다변수 함수. 각 파라미터에 대한 편미분이 필요.

In [ ]:
# 편미분 시각화: f(x, y) = x^2 + y^2
def f_2d(x, y):
    return x**2 + y**2

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 3D surface
ax = fig.add_subplot(131, projection='3d')
X, Y = np.meshgrid(np.linspace(-3, 3, 50), np.linspace(-3, 3, 50))
Z = f_2d(X, Y)
ax.plot_surface(X, Y, Z, cmap=cm.coolwarm, alpha=0.7)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('f(x,y)')
ax.set_title('f(x,y) = x^2 + y^2')

# y=1로 고정: f(x) 슬라이스
ax = axes[1]
x_vals = np.linspace(-3, 3, 100)
ax.plot(x_vals, f_2d(x_vals, 1), 'b-', linewidth=2, label='f(x, y=1)')
ax.plot(x_vals, 2 * x_vals, 'r--', linewidth=2, label='df/dx = 2x')
ax.set_xlabel('x')
ax.set_title('Partial wrt x (y=1 fixed)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(0, color='black', linewidth=0.5)

# x=1로 고정: f(y) 슬라이스
ax = axes[2]
y_vals = np.linspace(-3, 3, 100)
ax.plot(y_vals, f_2d(1, y_vals), 'b-', linewidth=2, label='f(x=1, y)')
ax.plot(y_vals, 2 * y_vals, 'r--', linewidth=2, label='df/dy = 2y')
ax.set_xlabel('y')
ax.set_title('Partial wrt y (x=1 fixed)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

---
## 2. Gradient

### 2.1 Gradient 벡터

모든 편미분을 벡터로 모은 것:

$$\nabla f = \begin{bmatrix} \frac{\partial f}{\partial x_1} \\ \frac{\partial f}{\partial x_2} \\ \vdots \end{bmatrix}$$

**핵심 성질**: Gradient 벡터는 함수가 **가장 가파르게 증가하는 방향**을 가리킨다.

- $\nabla f$ 방향: 가장 빠르게 증가
- $-\nabla f$ 방향: 가장 빠르게 감소 (Gradient Descent가 이 방향으로 이동!)

In [ ]:
# Gradient가 가장 가파른 상승 방향임을 시각화
def f_2d(x, y):
    return x**2 + 2*y**2

def grad_f(x, y):
    return np.array([2*x, 4*y])

fig, ax = plt.subplots(figsize=(8, 8))

# 등고선
X, Y = np.meshgrid(np.linspace(-3, 3, 100), np.linspace(-3, 3, 100))
Z = f_2d(X, Y)
contour = ax.contour(X, Y, Z, levels=15, cmap='coolwarm')
ax.clabel(contour, inline=True, fontsize=8)

# 여러 점에서의 Gradient 벡터 (화살표)
points = [(-2, -1.5), (-1, 2), (2, 1), (1, -1), (-2, 1)]
for (px, py) in points:
    g = grad_f(px, py)
    # 시각화를 위해 스케일 조정
    g_norm = g / np.linalg.norm(g) * 0.5
    ax.quiver(px, py, g_norm[0], g_norm[1], angles='xy', scale_units='xy',
              scale=1, color='red', linewidth=2)

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Gradient vectors on contour plot\n(Red arrows = direction of steepest ascent)')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.show()

print("Gradient 벡터(빨간 화살표)는 항상 등고선에 수직이다!")
print("-> 등고선에 수직 = 함수값이 가장 빠르게 변하는 방향")

---
## 3. Gradient Descent

ML 학습의 핵심 알고리즘. 손실 함수를 최소화하기 위해 gradient의 **반대 방향**으로 이동.

$$\theta_{t+1} = \theta_t - \alpha \nabla L(\theta_t)$$

- $\theta$: 파라미터 (가중치)
- $\alpha$: 학습률 (Learning Rate)
- $\nabla L$: 손실 함수의 gradient

### 3.1 1D Gradient Descent

In [ ]:
# 1D Gradient Descent: f(x) = x^2 + 2*sin(2x) (여러 극값이 있는 함수)
def f(x):
    return x**2 + 2 * np.sin(2 * x)

def f_grad(x):
    return 2 * x + 4 * np.cos(2 * x)

# Gradient Descent 실행
lr = 0.1
x = 3.0  # 시작점
history = [x]

for i in range(30):
    grad = f_grad(x)
    x = x - lr * grad
    history.append(x)

history = np.array(history)

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 함수 위에 경로 표시
ax = axes[0]
x_plot = np.linspace(-4, 4, 200)
ax.plot(x_plot, f(x_plot), 'b-', linewidth=2, label='f(x)')
ax.plot(history, f(history), 'ro-', markersize=4, linewidth=1, label='GD path')
ax.plot(history[0], f(history[0]), 'g*', markersize=15, label='Start')
ax.plot(history[-1], f(history[-1]), 'r*', markersize=15, label=f'End (x={history[-1]:.3f})')
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.set_title(f'1D Gradient Descent (lr={lr})')
ax.legend()
ax.grid(True, alpha=0.3)

# 오른쪽: f(x) 값의 변화
ax = axes[1]
ax.plot(f(history), 'b-o', markersize=3)
ax.set_xlabel('Step')
ax.set_ylabel('f(x)')
ax.set_title('Loss over Steps')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"시작: x = 3.000, f(x) = {f(3.0):.4f}")
print(f"최종: x = {history[-1]:.4f}, f(x) = {f(history[-1]):.4f}")

### 3.2 2D Gradient Descent + Contour Plot

실제 ML에서는 파라미터가 수백만 개이지만, 2D로 시각화하면 직관을 얻을 수 있다.

In [ ]:
# 2D Gradient Descent 시각화
def loss(w):
    """Rosenbrock-like function: 최적화하기 어려운 함수"""
    x, y = w
    return (1 - x)**2 + 10 * (y - x**2)**2

def loss_grad(w):
    x, y = w
    dx = -2 * (1 - x) + 10 * 2 * (y - x**2) * (-2 * x)
    dy = 10 * 2 * (y - x**2)
    return np.array([dx, dy])

# Gradient Descent
def gradient_descent_2d(start, lr, n_steps):
    w = np.array(start, dtype=float)
    path = [w.copy()]
    for _ in range(n_steps):
        grad = loss_grad(w)
        w = w - lr * grad
        path.append(w.copy())
    return np.array(path)

# 등고선 그리드
X, Y = np.meshgrid(np.linspace(-2, 2, 200), np.linspace(-1, 3, 200))
Z = (1 - X)**2 + 10 * (Y - X**2)**2

fig, ax = plt.subplots(figsize=(10, 8))
contour = ax.contour(X, Y, Z, levels=np.logspace(-1, 3, 20), cmap='coolwarm')
ax.contourf(X, Y, Z, levels=np.logspace(-1, 3, 20), cmap='coolwarm', alpha=0.3)

# 시작점 (-1.5, 2.0)에서 Gradient Descent
path = gradient_descent_2d([-1.5, 2.0], lr=0.002, n_steps=500)
ax.plot(path[:, 0], path[:, 1], 'k.-', markersize=2, linewidth=1, label='GD path')
ax.plot(path[0, 0], path[0, 1], 'g*', markersize=15, label='Start')
ax.plot(1, 1, 'r*', markersize=15, label='Global minimum (1, 1)')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('2D Gradient Descent on Rosenbrock Function')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.show()

print(f"시작: ({path[0, 0]:.1f}, {path[0, 1]:.1f}), loss = {loss(path[0]):.4f}")
print(f"최종: ({path[-1, 0]:.4f}, {path[-1, 1]:.4f}), loss = {loss(path[-1]):.6f}")
print(f"최적점: (1, 1), loss = {loss([1, 1])}")

### 3.3 Learning Rate의 영향

학습률이 너무 크면 발산, 너무 작으면 느리게 수렴.

- **너무 작은 lr**: 수렴은 하지만 매우 느림
- **적절한 lr**: 빠르고 안정적으로 수렴
- **너무 큰 lr**: 진동하거나 발산

In [ ]:
# Learning Rate 비교: f(x) = x^2
def f(x):
    return x ** 2

def f_grad(x):
    return 2 * x

learning_rates = [0.01, 0.1, 0.5, 0.95]
start = 3.0
n_steps = 20

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
x_plot = np.linspace(-4, 4, 200)

for ax, lr in zip(axes, learning_rates):
    x = start
    history = [x]
    for _ in range(n_steps):
        x = x - lr * f_grad(x)
        history.append(x)
    history = np.array(history)

    ax.plot(x_plot, f(x_plot), 'b-', linewidth=2)
    ax.plot(history, f(history), 'ro-', markersize=4, linewidth=1)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-1, 16)
    ax.set_title(f'lr = {lr}\nfinal x = {history[-1]:.4f}')
    ax.grid(True, alpha=0.3)

plt.suptitle('Effect of Learning Rate on Gradient Descent', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

print("lr=0.01: 너무 느림 (20스텝 후에도 최솟값에 도달 못함)")
print("lr=0.1:  적절함 (빠르게 수렴)")
print("lr=0.5:  수렴하지만 0에서 정확히 멈춤 (x^2의 특수 경우)")
print("lr=0.95: 진동하면서 느리게 수렴 (위험!)")

In [ ]:
# 2D에서 Learning Rate 비교
def simple_loss(w):
    return w[0]**2 + 5 * w[1]**2

def simple_loss_grad(w):
    return np.array([2 * w[0], 10 * w[1]])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

X, Y = np.meshgrid(np.linspace(-3, 3, 100), np.linspace(-3, 3, 100))
Z = X**2 + 5 * Y**2

lrs = [0.01, 0.1, 0.19]
labels = ['Too small', 'Good', 'Too large']

for ax, lr, label in zip(axes, lrs, labels):
    ax.contour(X, Y, Z, levels=15, cmap='coolwarm', alpha=0.5)

    w = np.array([2.5, 2.5])
    path = [w.copy()]
    for _ in range(30):
        w = w - lr * simple_loss_grad(w)
        path.append(w.copy())
    path = np.array(path)

    ax.plot(path[:, 0], path[:, 1], 'ro-', markersize=3, linewidth=1)
    ax.plot(path[0, 0], path[0, 1], 'g*', markersize=12)
    ax.set_title(f'{label}\nlr={lr}')
    ax.set_xlabel('w1')
    ax.set_ylabel('w2')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Chain Rule (연쇄 법칙)

### 4.1 합성함수의 미분

$$\frac{d}{dx} f(g(x)) = f'(g(x)) \cdot g'(x)$$

여러 함수가 겹쳐있을 때, 각 단계의 미분을 **곱하면** 된다.

### 역전파(Backpropagation)의 수학적 기초

신경망은 합성함수:
$$\text{Loss} = L(\sigma(W_2 \cdot \sigma(W_1 \cdot x + b_1) + b_2))$$

Chain Rule로 각 가중치에 대한 미분을 구할 수 있다!

In [ ]:
# Chain Rule 예제: f(x) = (3x + 2)^2
# g(x) = 3x + 2, h(u) = u^2
# f(x) = h(g(x))
# f'(x) = h'(g(x)) * g'(x) = 2*(3x+2) * 3 = 6*(3x+2)

x = np.linspace(-3, 3, 200)

# 직접 미분 (전개 후 미분)
# f(x) = 9x^2 + 12x + 4 -> f'(x) = 18x + 12
direct = 18 * x + 12

# Chain Rule로 미분
chain = 6 * (3 * x + 2)

print("f(x) = (3x + 2)^2")
print("직접 전개 후 미분: f'(x) = 18x + 12")
print("Chain Rule: f'(x) = 2*(3x+2) * 3 = 6*(3x+2) = 18x + 12")
print(f"두 방법이 같은가? {np.allclose(direct, chain)}")

In [ ]:
# 역전파 시뮬레이션: 간단한 2-layer network
# Forward: x -> [*w1 + b1] -> [sigmoid] -> [*w2 + b2] -> [MSE loss]

# 파라미터
x = 2.0     # 입력
y = 1.0     # 정답
w1 = 0.5
b1 = 0.1
w2 = -0.3
b2 = 0.2

# === Forward Pass ===
z1 = w1 * x + b1           # 선형 변환 1
a1 = 1 / (1 + np.exp(-z1)) # sigmoid
z2 = w2 * a1 + b2          # 선형 변환 2
loss = (z2 - y) ** 2       # MSE loss

print("=== Forward Pass ===")
print(f"z1 = w1*x + b1 = {w1}*{x} + {b1} = {z1}")
print(f"a1 = sigmoid(z1) = {a1:.4f}")
print(f"z2 = w2*a1 + b2 = {w2}*{a1:.4f} + {b2} = {z2:.4f}")
print(f"loss = (z2 - y)^2 = ({z2:.4f} - {y})^2 = {loss:.4f}")

# === Backward Pass (Chain Rule) ===
# dL/dz2
dL_dz2 = 2 * (z2 - y)
# dL/dw2 = dL/dz2 * dz2/dw2
dL_dw2 = dL_dz2 * a1
# dL/db2 = dL/dz2 * dz2/db2
dL_db2 = dL_dz2 * 1
# dL/da1 = dL/dz2 * dz2/da1
dL_da1 = dL_dz2 * w2
# dL/dz1 = dL/da1 * da1/dz1
dL_dz1 = dL_da1 * a1 * (1 - a1)  # sigmoid 도함수
# dL/dw1 = dL/dz1 * dz1/dw1
dL_dw1 = dL_dz1 * x
# dL/db1 = dL/dz1 * dz1/db1
dL_db1 = dL_dz1 * 1

print("\n=== Backward Pass (Chain Rule) ===")
print(f"dL/dz2 = {dL_dz2:.4f}")
print(f"dL/dw2 = {dL_dw2:.4f}")
print(f"dL/db2 = {dL_db2:.4f}")
print(f"dL/da1 = {dL_da1:.4f}")
print(f"dL/dz1 = {dL_dz1:.4f}")
print(f"dL/dw1 = {dL_dw1:.4f}")
print(f"dL/db1 = {dL_db1:.4f}")

In [ ]:
# 역전파 과정을 Computational Graph로 시각화
fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(-0.5, 7.5)
ax.set_ylim(-1, 3)

# 노드 위치
nodes = {
    'x': (0, 2), 'w1': (0, 1), 'b1': (0, 0),
    'z1': (2, 1.5), 'a1': (3.5, 1.5),
    'w2': (3.5, 0), 'b2': (5, 0),
    'z2': (5.5, 1.5), 'L': (7, 1.5)
}

# 노드 그리기
for name, (nx, ny) in nodes.items():
    ax.plot(nx, ny, 'o', markersize=25, color='lightblue', markeredgecolor='black')
    ax.text(nx, ny, name, ha='center', va='center', fontsize=10, fontweight='bold')

# 연결선과 gradient
connections = [
    ('x', 'z1', f'w1={w1}'), ('w1', 'z1', f'x={x}'), ('b1', 'z1', '1'),
    ('z1', 'a1', f'sig\'={a1*(1-a1):.3f}'),
    ('a1', 'z2', f'w2={w2}'), ('w2', 'z2', f'a1={a1:.3f}'), ('b2', 'z2', '1'),
    ('z2', 'L', f'2(z2-y)={dL_dz2:.3f}')
]

for start, end, label in connections:
    sx, sy = nodes[start]
    ex, ey = nodes[end]
    ax.annotate('', xy=(ex - 0.15, ey), xytext=(sx + 0.15, sy),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
    mx, my = (sx + ex) / 2, (sy + ey) / 2 + 0.2
    ax.text(mx, my, label, fontsize=7, ha='center', color='blue')

ax.set_title('Computational Graph (Forward: gray arrows, values on edges)', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.show()

print("역전파 = 오른쪽(Loss)에서 왼쪽(파라미터)으로 Chain Rule 적용")
print("각 간선의 local gradient를 곱해나가면 된다!")

---
## 5. 수치 미분 vs 해석적 미분

### 5.1 수치 미분 (Numerical Differentiation)

도함수의 정의를 직접 근사:

$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h} \quad \text{(Central Difference)}$$

- 장점: 어떤 함수든 미분 가능 (공식 불필요)
- 단점: 느림 (파라미터마다 함수를 2번 계산), 근사값

### 5.2 해석적 미분 (Analytical Differentiation)

미분 공식을 사용하여 정확한 도함수를 구하는 것.

- 장점: 정확, 빠름
- 단점: 공식을 유도해야 함 (자동 미분이 해결!)

In [ ]:
# 수치 미분 구현
def numerical_gradient(f, x, h=1e-5):
    """Central difference로 수치 미분 계산"""
    grad = np.zeros_like(x)
    for i in range(len(x)):
        x_plus = x.copy()
        x_minus = x.copy()
        x_plus[i] += h
        x_minus[i] -= h
        grad[i] = (f(x_plus) - f(x_minus)) / (2 * h)
    return grad

# 테스트: f(x, y) = x^2 + 3xy + y^2
def f_test(w):
    x, y = w
    return x**2 + 3*x*y + y**2

def f_test_grad_analytical(w):
    x, y = w
    return np.array([2*x + 3*y, 3*x + 2*y])

# 비교
w = np.array([2.0, 3.0])
numerical = numerical_gradient(f_test, w)
analytical = f_test_grad_analytical(w)

print(f"점 w = {w}에서의 gradient:")
print(f"수치 미분:  {numerical}")
print(f"해석적 미분: {analytical}")
print(f"차이: {np.abs(numerical - analytical)}")
print(f"상대 오차: {np.abs(numerical - analytical) / np.abs(analytical)}")

In [ ]:
# h의 크기에 따른 수치 미분의 정확도
def f_simple(x):
    return x[0] ** 3  # f(x) = x^3, f'(x) = 3x^2

x = np.array([2.0])
true_grad = 12.0  # 3 * 2^2

h_values = np.logspace(-1, -15, 30)
errors_forward = []  # forward difference
errors_central = []  # central difference

for h in h_values:
    # Forward difference: (f(x+h) - f(x)) / h
    forward = ((x[0] + h)**3 - x[0]**3) / h
    errors_forward.append(abs(forward - true_grad))

    # Central difference: (f(x+h) - f(x-h)) / (2h)
    central = ((x[0] + h)**3 - (x[0] - h)**3) / (2 * h)
    errors_central.append(abs(central - true_grad))

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(h_values, errors_forward, 'b-o', markersize=3, label='Forward difference')
ax.loglog(h_values, errors_central, 'r-o', markersize=3, label='Central difference')
ax.set_xlabel('h')
ax.set_ylabel('Absolute error')
ax.set_title('Numerical Differentiation: Error vs Step Size h\n(f(x) = x^3 at x=2, true gradient = 12)')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.axvline(1e-5, color='green', linestyle='--', alpha=0.5, label='h = 1e-5')
ax.legend(fontsize=11)
plt.show()

print("h가 작아질수록 정확해지다가, 너무 작으면 부동소수점 오차로 다시 부정확해진다")
print("Central difference가 Forward difference보다 같은 h에서 더 정확하다")
print("h = 1e-5 ~ 1e-7 정도가 실용적인 선택")

### 5.3 Gradient Checking

역전파 구현이 올바른지 검증하는 실전 기법.

$$\text{relative error} = \frac{|g_{\text{analytical}} - g_{\text{numerical}}|}{\max(|g_{\text{analytical}}|, |g_{\text{numerical}}|)}$$

- relative error < $10^{-7}$: 정확
- relative error > $10^{-3}$: 버그가 있을 가능성 높음

In [ ]:
# Gradient Checking 실전 예제: 2-layer network
def forward_and_loss(params, x, y):
    """Simple 2-layer network: x -> linear -> sigmoid -> linear -> MSE"""
    w1, b1, w2, b2 = params
    z1 = w1 * x + b1
    a1 = 1 / (1 + np.exp(-z1))
    z2 = w2 * a1 + b2
    loss = (z2 - y) ** 2
    return loss

def backward(params, x, y):
    """해석적 gradient (역전파)"""
    w1, b1, w2, b2 = params
    z1 = w1 * x + b1
    a1 = 1 / (1 + np.exp(-z1))
    z2 = w2 * a1 + b2

    dL_dz2 = 2 * (z2 - y)
    dL_dw2 = dL_dz2 * a1
    dL_db2 = dL_dz2
    dL_da1 = dL_dz2 * w2
    dL_dz1 = dL_da1 * a1 * (1 - a1)
    dL_dw1 = dL_dz1 * x
    dL_db1 = dL_dz1

    return np.array([dL_dw1, dL_db1, dL_dw2, dL_db2])

# Gradient Checking
params = np.array([0.5, 0.1, -0.3, 0.2])
x, y = 2.0, 1.0

analytical_grad = backward(params, x, y)
numerical_grad = numerical_gradient(
    lambda p: forward_and_loss(p, x, y), params
)

print("Gradient Checking Results:")
print(f"{'Param':>6s} {'Analytical':>12s} {'Numerical':>12s} {'Rel Error':>12s} {'OK?':>5s}")
print("-" * 50)
names = ['w1', 'b1', 'w2', 'b2']
for name, a, n in zip(names, analytical_grad, numerical_grad):
    rel_error = abs(a - n) / max(abs(a), abs(n), 1e-8)
    ok = "OK" if rel_error < 1e-5 else "FAIL"
    print(f"{name:>6s} {a:>12.8f} {n:>12.8f} {rel_error:>12.2e} {ok:>5s}")

print("\n-> 모든 파라미터의 gradient가 일치! 역전파 구현이 정확하다.")

In [ ]:
# 의도적으로 잘못된 gradient로 gradient checking 실패 예시
def backward_buggy(params, x, y):
    """버그가 있는 역전파 (w1의 gradient에 버그)"""
    w1, b1, w2, b2 = params
    z1 = w1 * x + b1
    a1 = 1 / (1 + np.exp(-z1))
    z2 = w2 * a1 + b2

    dL_dz2 = 2 * (z2 - y)
    dL_dw2 = dL_dz2 * a1
    dL_db2 = dL_dz2
    dL_da1 = dL_dz2 * w2
    dL_dz1 = dL_da1 * a1 * (1 - a1)
    dL_dw1 = dL_dz1  # BUG: x를 곱하지 않음!
    dL_db1 = dL_dz1

    return np.array([dL_dw1, dL_db1, dL_dw2, dL_db2])

buggy_grad = backward_buggy(params, x, y)

print("Gradient Checking with BUGGY backward:")
print(f"{'Param':>6s} {'Buggy':>12s} {'Numerical':>12s} {'Rel Error':>12s} {'OK?':>5s}")
print("-" * 50)
for name, a, n in zip(names, buggy_grad, numerical_grad):
    rel_error = abs(a - n) / max(abs(a), abs(n), 1e-8)
    ok = "OK" if rel_error < 1e-5 else "FAIL"
    print(f"{name:>6s} {a:>12.8f} {n:>12.8f} {rel_error:>12.2e} {ok:>5s}")

print("\n-> w1의 gradient에서 FAIL! 버그를 잡았다!")

### 5.4 Gradient Descent 직접 구현: 선형 회귀

지금까지 배운 것을 종합하여 선형 회귀를 Gradient Descent로 학습해보자.

In [ ]:
# 데이터 생성: y = 3x + 2 + noise
np.random.seed(42)
n = 50
X = np.random.uniform(-3, 3, n)
y = 3 * X + 2 + np.random.normal(0, 0.5, n)

# 파라미터 초기화
w = 0.0  # weight (기울기)
b = 0.0  # bias (절편)
lr = 0.01
n_epochs = 100

# 학습 기록
loss_history = []
w_history = []
b_history = []

for epoch in range(n_epochs):
    # Forward: 예측
    y_pred = w * X + b

    # Loss: MSE
    loss = np.mean((y_pred - y) ** 2)
    loss_history.append(loss)
    w_history.append(w)
    b_history.append(b)

    # Backward: gradient 계산 (해석적 미분)
    dL_dw = np.mean(2 * (y_pred - y) * X)
    dL_db = np.mean(2 * (y_pred - y))

    # Update
    w = w - lr * dL_dw
    b = b - lr * dL_db

print(f"학습 완료!")
print(f"추정: y = {w:.4f}x + {b:.4f}")
print(f"실제: y = 3x + 2")
print(f"최종 Loss: {loss_history[-1]:.4f}")

In [ ]:
# 학습 과정 시각화
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Loss 변화
ax = axes[0]
ax.plot(loss_history, 'b-', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Training Loss')
ax.grid(True, alpha=0.3)

# 2. 데이터 + 학습된 선
ax = axes[1]
ax.scatter(X, y, alpha=0.5, s=20, label='Data')
x_line = np.linspace(-3, 3, 100)
ax.plot(x_line, 3 * x_line + 2, 'g-', linewidth=2, label='True: y=3x+2')
ax.plot(x_line, w * x_line + b, 'r--', linewidth=2,
        label=f'Learned: y={w:.2f}x+{b:.2f}')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Linear Regression Result')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Parameter space에서의 경로
ax = axes[2]
W_grid, B_grid = np.meshgrid(np.linspace(-1, 5, 100), np.linspace(-2, 4, 100))
L_grid = np.zeros_like(W_grid)
for i in range(W_grid.shape[0]):
    for j in range(W_grid.shape[1]):
        y_p = W_grid[i, j] * X + B_grid[i, j]
        L_grid[i, j] = np.mean((y_p - y) ** 2)
ax.contour(W_grid, B_grid, L_grid, levels=30, cmap='coolwarm', alpha=0.5)
ax.plot(w_history, b_history, 'ko-', markersize=2, linewidth=1)
ax.plot(w_history[0], b_history[0], 'g*', markersize=12, label='Start')
ax.plot(w_history[-1], b_history[-1], 'r*', markersize=12, label='End')
ax.set_xlabel('w (weight)')
ax.set_ylabel('b (bias)')
ax.set_title('Parameter Space Trajectory')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: Gradient Descent로 2차 함수 최솟값 찾기

아래 함수의 최솟값을 Gradient Descent로 찾으세요.

$$f(x, y) = (x-3)^2 + (y+1)^2 + 2xy$$

- 해석적 gradient를 직접 구하세요
- 시작점 (0, 0), lr = 0.05, 100 스텝
- 경로를 등고선 위에 시각화

In [ ]:
# TODO:
# 1. f(x, y)와 gradient 함수 정의
#    df/dx = 2(x-3) + 2y = 2x + 2y - 6
#    df/dy = 2(y+1) + 2x = 2x + 2y + 2
# 2. Gradient Descent 실행
# 3. 등고선 + 경로 시각화
# 4. 수치 미분으로 gradient 검증 (gradient checking)


### 연습 2: Sigmoid + Cross-Entropy의 Gradient

이진 분류에서 자주 쓰이는 조합:

$$L = -[y \log(\sigma(z)) + (1-y) \log(1-\sigma(z))]$$

여기서 $\sigma(z) = 1/(1+e^{-z})$

1. $dL/dz$를 해석적으로 구하세요 (힌트: 결과가 매우 깔끔합니다)
2. 수치 미분으로 검증하세요

In [ ]:
# TODO:
# 1. sigmoid, binary_cross_entropy 함수 정의
# 2. dL/dz를 해석적으로 구하세요
#    힌트: dL/dz = sigma(z) - y (놀랍도록 간단!)
# 3. 수치 미분으로 검증
# 4. z 값 범위에서 해석적 미분과 수치 미분을 그래프로 비교


### 연습 3: Learning Rate Scheduler 구현

고정 학습률 대신 학습이 진행될수록 lr을 줄이는 전략을 구현하고 비교하세요.

1. 고정 lr = 0.1
2. Step decay: 매 20 에폭마다 lr을 절반으로
3. Cosine annealing: $lr_t = lr_{min} + \frac{1}{2}(lr_{max} - lr_{min})(1 + \cos(\frac{t}{T}\pi))$

In [ ]:
# TODO:
# 1. 2차원 함수에 대해 Gradient Descent 실행 (3가지 lr 스케줄링)
# 2. 각 방법의 loss curve를 같은 그래프에 비교
# 3. 어떤 방법이 가장 빠르게 수렴하는지 관찰

def f_opt(w):
    return w[0]**2 + 5*w[1]**2 + 0.5*w[0]*w[1]

def f_opt_grad(w):
    return np.array([2*w[0] + 0.5*w[1], 10*w[1] + 0.5*w[0]])


---
## 핵심 정리

| 개념 | ML에서의 역할 |
|------|---------------|
| 도함수 | 파라미터 변화에 대한 손실 변화율 |
| 편미분 | 개별 파라미터에 대한 미분 |
| Gradient | 모든 편미분의 벡터, 가장 가파른 상승 방향 |
| Gradient Descent | 모델 학습의 핵심 알고리즘 |
| Learning Rate | 학습 속도 조절, 너무 크면 발산, 작으면 느림 |
| Chain Rule | 역전파(Backpropagation)의 수학적 기초 |
| 수치 미분 | Gradient Checking으로 역전파 구현 검증 |

**핵심 연결**: 역전파 = Chain Rule의 효율적 구현 = 모든 파라미터의 gradient를 한 번에 계산

**다음 노트북**: [04-numpy-pytorch-basics.ipynb](04-numpy-pytorch-basics.ipynb) - NumPy와 PyTorch 기초